# 1. Fine-tune XLM-RoBERTa cho task Token Classification
- NEERD_NORM (1)
- KEEP (0)

In [1]:
# ============================================================
# 1. Cài thư viện
# ============================================================
!pip install transformers torch

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [2]:

# ============================================================
# 2. Import
# ============================================================
import os
import re
import pickle
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer
)

/Users/nals_macbook_108/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/nals_macbook_108/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ============================================================
# 3. Cấu hình PATH
# Kaggle: /kaggle/input/...
# Local:  ../data/
# ============================================================
# ============================================================
# Cấu hình PATH — tự động detect Kaggle vs Local
# ============================================================
IS_KAGGLE = os.path.exists("/kaggle/working")

if IS_KAGGLE:
    DATA_PATH   = "/kaggle/input/multilex-norm-vi/"  # chỉnh lại nếu cần
    OUTPUT_PATH = "/kaggle/working/"
else:
    DATA_PATH   = "../data/"
    OUTPUT_PATH = "../outputs/"

train_path = DATA_PATH + "multilexnorm2026_vi_train.csv"
val_path   = DATA_PATH + "multilexnorm2026_vi_validation.csv"
test_path  = DATA_PATH + "multilexnorm2026_vi_test.csv"

os.makedirs(OUTPUT_PATH, exist_ok=True)

for path in [train_path, val_path, test_path]:
    print(path, "->", "FOUND" if os.path.exists(path) else "NOT FOUND")

print(f"\nMôi trường: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"DATA_PATH:   {DATA_PATH}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")

../data/multilexnorm2026_vi_train.csv -> FOUND
../data/multilexnorm2026_vi_validation.csv -> FOUND
../data/multilexnorm2026_vi_test.csv -> FOUND

Môi trường: Local
DATA_PATH:   ../data/
OUTPUT_PATH: ../outputs/


In [4]:
# ============================================================
# 4. Hàm parse token list
# Dùng cùng hàm với 01/02/03 để đảm bảo consistent
# ============================================================
def parse_token_list(text):
    text = str(text)
    tokens = re.findall(r"'([^']*)'", text)
    return tokens

In [5]:
# ============================================================
# 5. Đọc và parse dữ liệu
# ============================================================
train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

for df in [train_df, val_df, test_df]:
    df["raw_tokens"]  = df["raw"].apply(parse_token_list)
    df["norm_tokens"] = df["norm"].apply(parse_token_list)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Verify parse
print("\nVí dụ sau khi parse:")
print("RAW :", train_df.loc[0, "raw_tokens"])
print("NORM:", train_df.loc[0, "norm_tokens"])

Train: 8371 | Val: 1050 | Test: 522

Ví dụ sau khi parse:
RAW : ['thích', 'anh', 'cá', 'mập', 'k']
NORM: ['thích', 'anh', 'cá', 'mập', 'không']


In [6]:
# ============================================================
# 6. Detect device
# ============================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Device: GPU - {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Device: M1 MPS")
else:
    device = torch.device("cpu")
    print("Device: CPU")

Device: M1 MPS


In [7]:
# ============================================================
# 7. Load XLM-RoBERTa
# ============================================================
MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # 0=KEEP, 1=NEED_NORM
)
model = model.to(device)

print(f"Model: {MODEL_NAME}")
print(f"Số parameters: {sum(p.numel() for p in model.parameters()):,}")

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: xlm-roberta-base
Số parameters: 277,454,594


In [8]:
# ============================================================
# 8. Tạo label NEED_NORM/KEEP cho từng token
# ============================================================
def create_token_labels(row):
    """
    0 = KEEP (giữ nguyên)
    1 = NEED_NORM (cần normalize)
    """
    return [1 if r != n else 0
            for r, n in zip(row["raw_tokens"], row["norm_tokens"])]

train_df["labels"] = train_df.apply(create_token_labels, axis=1)
val_df["labels"]   = val_df.apply(create_token_labels, axis=1)
test_df["labels"]  = test_df.apply(create_token_labels, axis=1)

# Thống kê phân bố label
def count_labels(df, name):
    total     = sum(len(l) for l in df["labels"])
    need_norm = sum(sum(l) for l in df["labels"])
    keep      = total - need_norm
    print(f"=== {name} ===")
    print(f"Total:     {total:,}")
    print(f"NEED_NORM: {need_norm:,} ({need_norm/total:.2%})")
    print(f"KEEP:      {keep:,} ({keep/total:.2%})")
    print()

count_labels(train_df, "Train")
count_labels(val_df, "Val")

=== Train ===
Total:     101,773
NEED_NORM: 16,343 (16.06%)
KEEP:      85,430 (83.94%)

=== Val ===
Total:     13,649
NEED_NORM: 2,135 (15.64%)
KEEP:      11,514 (84.36%)



In [9]:

# ============================================================
# 9. Tokenize và align labels với subwords
# ============================================================
def tokenize_and_align_labels(raw_tokens, word_labels, tokenizer, max_length=128):
    """
    Subword cùng từ → cùng label
    Token đặc biệt ([CLS], [SEP], [PAD]) → label = -100
    """
    tokenized = tokenizer(
        raw_tokens,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt"
    )

    word_ids = tokenized.word_ids(0)
    aligned_labels = [
        -100 if word_idx is None else word_labels[word_idx]
        for word_idx in word_ids
    ]

    tokenized["labels"] = torch.tensor(aligned_labels)
    return tokenized

# Verify alignment
sample = train_df.iloc[0]
result = tokenize_and_align_labels(sample["raw_tokens"], sample["labels"], tokenizer)
tokens = tokenizer.convert_ids_to_tokens(result["input_ids"][0].tolist())
labels = result["labels"].tolist()

print(f"{'token':<15} {'label'}")
print("-" * 25)
for tok, lab in zip(tokens[:20], labels[:20]):
    print(f"{tok:<15} {lab}")


token           label
-------------------------
<s>             -100
▁thích          0
▁anh            0
▁cá             0
▁m              0
ập              0
▁k              1
</s>            -100
<pad>           -100
<pad>           -100
<pad>           -100
<pad>           -100
<pad>           -100
<pad>           -100
<pad>           -100
<pad>           -100
<pad>           -100
<pad>           -100
<pad>           -100
<pad>           -100


In [10]:

# ============================================================
# 10. Chuẩn bị dataset
# ============================================================
def prepare_dataset(df, tokenizer, max_length=128):
    all_input_ids, all_attention_masks, all_labels = [], [], []

    for _, row in df.iterrows():
        result = tokenize_and_align_labels(
            row["raw_tokens"], row["labels"], tokenizer, max_length
        )
        all_input_ids.append(result["input_ids"][0])
        all_attention_masks.append(result["attention_mask"][0])
        all_labels.append(result["labels"])

    return {
        "input_ids":      torch.stack(all_input_ids),
        "attention_mask": torch.stack(all_attention_masks),
        "labels":         torch.stack(all_labels)
    }

print("Tokenizing train set...")
train_data = prepare_dataset(train_df, tokenizer)
print(f"Train: {train_data['input_ids'].shape}")

print("Tokenizing val set...")
val_data = prepare_dataset(val_df, tokenizer)
print(f"Val: {val_data['input_ids'].shape}")

Tokenizing train set...
Train: torch.Size([8371, 128])
Tokenizing val set...
Val: torch.Size([1050, 128])


In [11]:
# Save để dùng lại nếu cần
with open(OUTPUT_PATH + "train_data.pkl", "wb") as f:
    pickle.dump(train_data, f)
with open(OUTPUT_PATH + "val_data.pkl", "wb") as f:
    pickle.dump(val_data, f)
print("Saved train_data.pkl, val_data.pkl")

Saved train_data.pkl, val_data.pkl


In [12]:
# ============================================================
# 11. PyTorch Dataset class
# ============================================================
class NormDataset(Dataset):
    def __init__(self, data):
        self.input_ids      = data["input_ids"]
        self.attention_mask = data["attention_mask"]
        self.labels         = data["labels"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx]
        }

train_dataset = NormDataset(train_data)
val_dataset   = NormDataset(val_data)

print(f"Train dataset: {len(train_dataset)}")
print(f"Val dataset:   {len(val_dataset)}")

Train dataset: 8371
Val dataset:   1050


In [13]:
# ============================================================
# 12a. Metrics
# ============================================================
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=-1)

    mask  = labels != -100
    preds = predictions[mask]
    labs  = labels[mask]

    TP = ((preds == 1) & (labs == 1)).sum()
    FP = ((preds == 1) & (labs == 0)).sum()
    FN = ((preds == 0) & (labs == 1)).sum()
    TN = ((preds == 0) & (labs == 0)).sum()

    accuracy  = (TP + TN) / len(labs) if len(labs) > 0 else 0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "accuracy":  float(accuracy),
        "precision": float(precision),
        "recall":    float(recall),
        "f1":        float(f1)
    }


In [14]:

# ============================================================
# 13. Training Arguments
# ============================================================
training_args = TrainingArguments(
    output_dir                  = OUTPUT_PATH + "xlmr_normalize_classifier",
    num_train_epochs            = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    logging_steps               = 100,
    report_to                   = "none"
)

trainer = Trainer(
    model          = model,
    args           = training_args,
    train_dataset  = train_dataset,
    eval_dataset   = val_dataset,
    compute_metrics= compute_metrics
)

print("Setup xong, sẵn sàng train!")

Setup xong, sẵn sàng train!


In [15]:
# ============================================================
# 14. Train
# ============================================================
trainer.train()

/Users/nals_macbook_108/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# ============================================================
# 15. Save best model
# ============================================================
MODEL_SAVE_PATH = OUTPUT_PATH + "xlmr_normalize_classifier"

trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)
print(f"Saved model to {MODEL_SAVE_PATH}")

In [ ]:
# Xóa checkpoint trung gian để giải phóng dung lượng
import shutil
for item in os.listdir(MODEL_SAVE_PATH):
    if item.startswith("checkpoint-"):
        shutil.rmtree(os.path.join(MODEL_SAVE_PATH, item))
        print(f"Đã xóa: {item}")

In [ ]:

# Zip để download về local
!cd /kaggle/working && zip -r xlmr_classifier.zip xlmr_normalize_classifier/
print("Done! Download xlmr_classifier.zip để dùng ở local.")